# Tutorial 5 — Unrestricted (UHF) references

The first four tutorials use a **restricted** (RHF) reference. This short
tutorial collects *what changes* when the reference is **unrestricted** (UHF),
so you can reuse the same AEGISS workflow on open-shell and spin-polarised
systems. It does not re-derive the theory — see the
[UHF reference page](../uhf.md) for the single-orbital-entropy formalism.

We use the **benzene radical cation** (`charge=1, spin=1`) as a genuinely
open-shell running example (α and β occupations differ).

## Differences at a glance

| Stage | Restricted (RHF) | Unrestricted (UHF) |
|---|---|---|
| Mean field | `run_scf(mol, method="RHF")` | `run_scf(mol, method="UHF")` |
| `mf.mo_coeff` | 2-D `(nao, nmo)` | 3-D `(2, nao, nmo)` — α/β blocks |
| `DMRGConfig` | any `symm_type` | **`symm_type="SZ"` required**; `unrestricted` autodetects; `spin_soe` controls SOE |
| FCIDUMP | restricted | `IUHF=1` (αα/ββ/αβ blocks) |
| Single-orbital entropy | 4-state spatial `S(1)` | spin-**summed** (same 4-state) or spin-**resolved** per-spin-orbital `H(n_α)`,`H(n_β)` |
| `select_by_ao_projection` | `np.ndarray` | split `{"alpha", "beta"}` dict |
| `restricted_space` / `contribution` | scalar / list | may also be a per-channel `{"alpha", "beta"}` dict |
| `select_aegiss` | `np.ndarray` | split `{"alpha", "beta"}` dict (α and β **not** reconciled) |

## 1. Imports

In [ ]:
import numpy as np

from aegiss import DMRGConfig, EntropyConfig, ProjectionConfig, OutputConfig
from aegiss.active_space.ao_projection import select_by_ao_projection
from aegiss.initial_guess.scf import build_mol, run_scf

## 2. Mean field — the only structural change

Build an **open-shell** molecule (`charge`/`spin` flow straight through to
PySCF) and ask for a `"UHF"` mean field. The single observable difference is
that `mf.mo_coeff` becomes **3-D** `(2, nao, nmo)` — separate α and β blocks.
Everything downstream keys off that shape.

In [1]:
mol = build_mol(
    "./data/benzene.xyz", basis="sto-3g", symmetry=False, verbose=0,
    charge=1, spin=1,          # radical cation: one unpaired electron
)
mf = run_scf(mol, method="UHF")

print("E(UHF)          =", round(float(mf.e_tot), 6))
print("<S^2>           =", round(float(mf.spin_square()[0]), 4))
print("mo_coeff shape  =", np.asarray(mf.mo_coeff).shape, "(RHF would be 2-D)")
print("nelec (na, nb)  =", mol.nelec)

E(UHF)          = -227.654904
<S^2>           = 0.9284
mo_coeff shape  = (2, 36, 36) (RHF would be 2-D)
nelec (na, nb)  = (21, 20)


## 3. `DMRGConfig` for UHF

Three fields matter for an unrestricted run:

- **`symm_type="SZ"`** — required. UHF breaks $\hat S^2$ symmetry, so the
  spin-adapted `"SU2"` solver is rejected.
- **`unrestricted`** — `None` (default) autodetects UHF from the mean field;
  `True`/`False` overrides. When active it writes an `IUHF=1` FCIDUMP.
- **`spin_soe`** — how single-orbital entropies are computed:
  `"summed"` (4-state spatial `S(1)` → one spatial space),
  `"resolved"` (per-spin-orbital `H(n_α)`, `H(n_β)` → α and β screened
  independently), or `"auto"` (picks `"resolved"` when `nα ≠ nβ`).

Resolve the effective choice with `DMRGConfig.spin_resolved()`.

In [2]:
dmrg_config = DMRGConfig(
    ncas=20, nelecas=(11, 10),  # open-shell CAS: (n_alpha, n_beta); na - nb = mol.spin
    symm_type="SZ",             # required for UHF
    unrestricted=None,          # autodetect from mf (UHF -> IUHF=1 FCIDUMP)
    spin_soe="auto",            # -> "resolved" here because na != nb
)
print("spin-resolved SOE?", dmrg_config.spin_resolved())

spin-resolved SOE? True


## 4. Single-orbital entropy: summed vs resolved

With `spin_soe="summed"` the entropy is the usual 4-state spatial `S(1)` and
screening yields **one** orbital list. With `spin_soe="resolved"` AEGISS uses
the per-spin-orbital binary entropy

$$ H(n_{i\sigma}) = -n_{i\sigma}\ln n_{i\sigma} - (1-n_{i\sigma})\ln(1-n_{i\sigma}), $$

computed from the 1-PDM diagonal (exact for a single spin-orbital by
particle-number symmetry), and screens the **α and β channels separately**.
The two are related by subadditivity
$\;|H_i^\alpha - H_i^\beta| \le S_i^{\text{spatial}} \le H_i^\alpha + H_i^\beta$;
the resolved form discards the intra-orbital α–β (double-occupancy) correlation.
See the [UHF reference page](../uhf.md#spin-summed-vs-spin-resolved-soe) for the
full treatment and a block2 cross-check.

## 5. AO projection returns a split dict

`select_by_ao_projection` detects the 3-D `mo_coeff` and screens each spin
block against the **same** (spin-independent) AO overlaps, returning a
`{"alpha", "beta"}` dict. This step needs no DMRG, so it runs here directly.

In [3]:
result = select_by_ao_projection(
    mf, ao_labels=["C 2pz"],
    projected_basis="minao",
    contribution=0.5, contribution_type="r",
)
print("type :", type(result).__name__)
print("alpha:", result["alpha"].tolist())
print("beta :", result["beta"].tolist())

type : dict
alpha: [16, 19, 20, 22, 23]
beta : [18, 19, 20, 21, 22, 23]


### Per-channel thresholds and windows

For a UHF reference both `contribution` and `restricted_space` accept a
per-channel `{"alpha", "beta"}` dict, letting you cut the two spin channels
differently (a non-dict value is shared by both). Here β is held to a stricter
relative cut than α:

In [4]:
result = select_by_ao_projection(
    mf, ao_labels=["C 2pz"],
    projected_basis="minao",
    contribution={"alpha": 0.5, "beta": 0.8},   # stricter cut on beta
    contribution_type="r",
)
print("alpha:", result["alpha"].tolist())
print("beta :", result["beta"].tolist())

alpha: [16, 19, 20, 22, 23]
beta : [18]


## 6. Full pipeline — `select_aegiss`

`select_aegiss` wires DMRG + entropy + AO projection together. For a UHF
reference it returns the same split `{"alpha", "beta"}` mapping; the α and β
selections are kept **separate** (an orbital index means a different MO in each
channel, so they are deliberately not reconciled).

This cell drives a DMRG solve through the `block2main` subprocess, so run it
in an environment where block2 is installed (it is not executed when building
these docs):

In [ ]:
from aegiss.active_space import select_aegiss

active = select_aegiss(
    mol, mf,
    dmrg_config=dmrg_config,
    entropy_config=EntropyConfig(fraction=0.1, fraction_type="r", ranking="global"),
    projection_config=ProjectionConfig(ao_labels=["C 2pz"], contribution=0.5),
    output_config=OutputConfig(saving_folder="./orbs/"),
)
# active == {"alpha": np.ndarray, "beta": np.ndarray}
print({k: v.tolist() for k, v in active.items()})

## Recap

Going unrestricted touches only the edges of the workflow:

1. `run_scf(..., method="UHF")` → 3-D `mo_coeff`.
2. `DMRGConfig(symm_type="SZ", spin_soe=...)` — everything else autodetects.
3. Selection functions return a split `{"alpha", "beta"}` dict instead of one
   array, and per-channel `contribution` / `restricted_space` dicts are available.

The α and β spaces are two independent index lists, never merged: an
unrestricted CAS needs as many α as β orbitals, so check the two lengths
before building it (see [The UHF active space](../uhf.md#the-uhf-active-space)).
For the entropy theory and a
numerical validation against block2, see the [UHF reference page](../uhf.md).